# Day 06: PyTorch, Model File Formats, ONNX & TensorRT
> *100 Days of Inference* | Layer: **Runtime** | Book: *Inference Engineering* Ch 4.2 (pp. 101–105)

**Prerequisite:** Day 07 (CUDA kernels)

**Goal:** Understand how model weights are stored, why file format matters for inference, and what ONNX and TensorRT bring to the table.

## What problem does this solve?

You've trained or downloaded a model. Now you want to deploy it. How do you store 70 billion parameters across multiple files? How do you load them safely and fast? How do you convert a PyTorch model to run on a TensorRT-optimized engine?

This is just like software packaging: you need a format that is safe to deserialize, portable across systems, and fast to load. The difference is scale — a single model can be 140 GB in FP16. Loading it wrong costs minutes of startup time every cold start.

Understanding model formats is also key to understanding the inference stack: the path from `model.safetensors` on disk to CUDA kernels running on the GPU passes through several layers, each with its own tradeoffs.

## Concept Overview

**PyTorch** is the dominant deep learning framework — every major open model is a PyTorch model. It handles both training and inference.

**Model file formats** store the weights (parameters) — the billions of numbers that define the model's behavior:

| Format | What it stores | Key property |
|--------|---------------|-------------|
| **safetensors** | Weights only | Safe (no code execution), fast memory-mapped loading |
| **ONNX** | Weights + computation graph | Portable across hardware, exportable from PyTorch |
| **.pt / .bin** | Weights (+ sometimes code) | Legacy, uses pickle (unsafe, can execute code) |

**ONNX Runtime** and **TensorRT** are inference runtimes that take a model and compile it to hardware-optimized code:
- ONNX Runtime: cross-platform, open standard
- TensorRT: NVIDIA-only, highest performance on NVIDIA GPUs

**Infrastructure analogy:** safetensors are like a static binary blob (just data). ONNX is like a portable container image. TensorRT is like a hardware-specific compiled binary — fastest to run, but must be recompiled for each GPU architecture.

In [3]:
!pip install -q numpy matplotlib torch safetensors 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10
GPU Memory: 128.5 GB


## Part 1: PyTorch — Tensor Operations and Model Representation

PyTorch represents models as collections of **tensors** (n-dimensional arrays) organized into `nn.Module` objects. Every model has:
- **Parameters:** The learned weights (stored as tensors, updated during training)
- **Buffers:** Non-learned state (e.g., running statistics in BatchNorm)
- **Forward function:** The computation graph (how inputs become outputs)

**Key insight:** For inference, you only need the *parameters* — the computation graph is defined by the code. This is why safetensors can store just weights without code.

In [4]:
# Build a small transformer-like model and inspect its weight structure

class MiniTransformerBlock(nn.Module):
    """A simplified transformer block: attention + FFN (no actual attention for simplicity)."""
    def __init__(self, hidden=256, heads=4, ffn_mult=4):
        super().__init__()
        self.q_proj = nn.Linear(hidden, hidden, bias=False)
        self.k_proj = nn.Linear(hidden, hidden, bias=False)
        self.v_proj = nn.Linear(hidden, hidden, bias=False)
        self.o_proj = nn.Linear(hidden, hidden, bias=False)
        self.ffn_gate = nn.Linear(hidden, hidden * ffn_mult, bias=False)
        self.ffn_up   = nn.Linear(hidden, hidden * ffn_mult, bias=False)
        self.ffn_down  = nn.Linear(hidden * ffn_mult, hidden, bias=False)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

model = MiniTransformerBlock(hidden=1024, heads=8)

# Inspect weight sizes
total_params = 0
print(f"{'Layer':<25} {'Shape':>20} {'Params':>10} {'Size (MB)':>10}")
print("-" * 70)
for name, param in model.named_parameters():
    n_params = param.numel()
    size_mb = n_params * param.element_size() / 1e6
    total_params += n_params
    print(f"{name:<25} {str(list(param.shape)):>20} {n_params:>10,} {size_mb:>9.2f} MB")
print("-" * 70)
total_fp16_mb = total_params * 2 / 1e6
total_fp32_mb = total_params * 4 / 1e6
print(f"{'Total':>47} {total_params:>10,}")
print(f"\nMemory footprint:")
print(f"  FP32 (4 bytes/param): {total_fp32_mb:.1f} MB")
print(f"  FP16 (2 bytes/param): {total_fp16_mb:.1f} MB")

# Extrapolate to real model sizes
print(f"\nReal model estimates (FP16):")
for name, billions in [("7B model", 7), ("13B model", 13), ("70B model", 70), ("671B DeepSeek", 671)]:
    gb = billions * 1e9 * 2 / 1e9
    print(f"  {name}: {gb:.0f} GB")

Layer                                    Shape     Params  Size (MB)
----------------------------------------------------------------------
q_proj.weight                     [1024, 1024]  1,048,576      4.19 MB
k_proj.weight                     [1024, 1024]  1,048,576      4.19 MB
v_proj.weight                     [1024, 1024]  1,048,576      4.19 MB
o_proj.weight                     [1024, 1024]  1,048,576      4.19 MB
ffn_gate.weight                   [4096, 1024]  4,194,304     16.78 MB
ffn_up.weight                     [4096, 1024]  4,194,304     16.78 MB
ffn_down.weight                   [1024, 4096]  4,194,304     16.78 MB
norm1.weight                            [1024]      1,024      0.00 MB
norm1.bias                              [1024]      1,024      0.00 MB
norm2.weight                            [1024]      1,024      0.00 MB
norm2.bias                              [1024]      1,024      0.00 MB
----------------------------------------------------------------------
        

## Part 2: safetensors — The Dominant Weight Format

**safetensors** is the current standard for storing model weights. Created by Hugging Face, it replaces the older `.bin` (pickle) format.

Key properties:
- **Safe:** Unlike pickle, safetensors cannot execute arbitrary code during loading — it's pure data
- **Fast:** Uses memory mapping — the OS maps the file to virtual memory; tensors are loaded lazily as needed
- **Sharded:** Large models are split across multiple files (e.g., `model-00001-of-00008.safetensors`)

**Infrastructure analogy:** Compare safetensors to a static binary artifact vs. a script:
- `.bin` (pickle) = a shell script that installs software — functional but could run anything
- safetensors = a pre-compiled binary — just data, safe to run anywhere

The sharding strategy matters for cold starts: loading 8 shards in parallel is much faster than loading 1 sequential file.

In [5]:
import tempfile
import os
import time

try:
    from safetensors.torch import save_file, load_file
    HAS_SAFETENSORS = True
except ImportError:
    HAS_SAFETENSORS = False
    print("safetensors not installed, showing format comparison only")

# Reuse model from the previous cell
state_dict = model.state_dict()

with tempfile.TemporaryDirectory() as tmpdir:
    pt_path = os.path.join(tmpdir, "model.pt")
    st_path = os.path.join(tmpdir, "model.safetensors")

    # Save as PyTorch .pt (pickle-based)
    torch.save(state_dict, pt_path)
    pt_size = os.path.getsize(pt_path)

    # Time loading .pt
    t = time.perf_counter()
    for _ in range(10):
        sd = torch.load(pt_path, map_location='cpu', weights_only=True)
    t_pt = (time.perf_counter() - t) / 10 * 1000

    if HAS_SAFETENSORS:
        # Save as safetensors
        save_file(state_dict, st_path)
        st_size = os.path.getsize(st_path)

        # Time loading safetensors
        t = time.perf_counter()
        for _ in range(10):
            sd = load_file(st_path, device='cpu')
        t_st = (time.perf_counter() - t) / 10 * 1000

        print(f"Format comparison:")
        print(f"  PyTorch .pt:    {pt_size/1e6:.1f} MB, load time: {t_pt:.1f} ms")
        print(f"  safetensors:    {st_size/1e6:.1f} MB, load time: {t_st:.1f} ms")
        print(f"  Speedup: {t_pt/t_st:.2f}x (for small model; scales dramatically at 70B+)")
    else:
        print(f"PyTorch .pt: {pt_size/1e6:.1f} MB, load time: {t_pt:.1f} ms")

print(f"\nSharding example (large model):")
print(f"  70B FP16 model = 140 GB")
print(f"  Typical: split into 8 shards x 17.5 GB")
print(f"  Loading 8 files in parallel = ~8x faster cold start")

Format comparison:
  PyTorch .pt:    67.1 MB, load time: 11.3 ms
  safetensors:    67.1 MB, load time: 0.2 ms
  Speedup: 52.47x (for small model; scales dramatically at 70B+)

Sharding example (large model):
  70B FP16 model = 140 GB
  Typical: split into 8 shards x 17.5 GB
  Loading 8 files in parallel = ~8x faster cold start


## Part 3: ONNX — Portable Model Graphs

**ONNX** (Open Neural Network Exchange) stores both the computation graph AND the weights. Unlike safetensors (weights only), ONNX captures the mathematical operations so that any compatible runtime can execute the model.

The ONNX format uses a standardized set of **operators** (like a portable bytecode). Any runtime that implements these operators can run the model, regardless of the original framework.

**Infrastructure analogy:** ONNX is like a Docker image — it bundles everything needed to run the model, not just the binary. You can run it on any compatible container runtime (ONNX Runtime, TensorRT, etc.).

Limitations:
- Not all PyTorch operations have ONNX equivalents
- Complex custom operations (like Multi-Latent Attention in DeepSeek) may not export cleanly
- Export process requires tracing the model, which can fail with dynamic control flow

In [6]:
!pip install -q onnx onnxscript 2>/dev/null

# Export a model to ONNX and inspect the graph structure
import io

class SimpleFFN(nn.Module):
    """Simple FFN block suitable for ONNX export."""
    def __init__(self, hidden=512, ffn=2048):
        super().__init__()
        self.fc1 = nn.Linear(hidden, ffn)
        self.fc2 = nn.Linear(ffn, hidden)
        self.norm = nn.LayerNorm(hidden)

    def forward(self, x):
        residual = x
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.fc2(x)
        return self.norm(x + residual)

model = SimpleFFN(hidden=512, ffn=2048)
model.eval()

# Export to ONNX
dummy_input = torch.randn(4, 32, 512)  # batch=4, seq=32, hidden=512
buffer = io.BytesIO()

try:
    torch.onnx.export(
        model,
        dummy_input,
        buffer,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch', 1: 'seq'}, 'output': {0: 'batch', 1: 'seq'}},
        opset_version=17,
    )
    onnx_size = buffer.tell()
    print(f"ONNX export successful!")
    print(f"  ONNX model size: {onnx_size/1e6:.2f} MB")
    print(f"  Includes weights + computation graph")

    # Check with onnx if available
    try:
        import onnx
        buffer.seek(0)
        model_onnx = onnx.load(buffer)
        print(f"  Number of ONNX nodes: {len(model_onnx.graph.node)}")
        node_types = {}
        for node in model_onnx.graph.node:
            node_types[node.op_type] = node_types.get(node.op_type, 0) + 1
        print(f"  Op types: {dict(node_types)}")
    except ImportError:
        print("  (install onnx to inspect graph structure)")

except Exception as e:
    print(f"ONNX export: {e}")
    print("Note: ONNX export of complex models (with custom ops, dynamic control flow) can fail.")

print(f"\nWhen to use ONNX:")
print(f"  + Cross-platform portability (ONNX Runtime supports AMD, Intel, ARM)")
print(f"  + TensorRT compilation pipeline")
print(f"  - Complex models with custom ops may not export cleanly")
print(f"  - Modern trend: skip ONNX, go PyTorch -> safetensors -> vLLM/TRT-LLM directly")

/tmp/ipykernel_3994187/1788397162.py:29: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0413 22:18:14.355000 3994187 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0413 22:18:14.560000 3994187 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0413 22:18:14.561000 3994187 torch/onnx/_internal/exporter/_registration.py:110

[torch.onnx] Obtain model graph for `SimpleFFN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleFFN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX export successful!
  ONNX model size: 8.40 MB
  Includes weights + computation graph
  Number of ONNX nodes: 8
  Op types: {'MatMul': 2, 'Add': 3, 'Sigmoid': 1, 'Mul': 1, 'LayerNormalization': 1}

When to use ONNX:
  + Cross-platform portability (ONNX Runtime supports AMD, Intel, ARM)
  + TensorRT compilation pipeline
  - Complex models with custom ops may not export cleanly
  - Modern trend: skip ONNX, go PyTorch -> safetensors -> vLLM/TRT-LLM directly


## Part 4: torch.compile — Automatic Graph Optimization

`torch.compile` is PyTorch's built-in compilation step. It analyzes your model's computation graph and applies optimizations like:
- **Kernel fusion:** Merge consecutive operations to reduce memory trips
- **Automatic kernel selection:** Pick the best kernel for the target hardware
- **Dead code elimination:** Remove unused operations

**Limitation:** `torch.compile` cannot fuse kernels that require custom CUDA code (like FlashAttention). Those require handwritten fused kernels, which is why production inference engines still manually manage kernel selection.

In [7]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class FFNBlock(nn.Module):
    def __init__(self, hidden=2048, ffn=8192):
        super().__init__()
        self.gate = nn.Linear(hidden, ffn, bias=False)
        self.up = nn.Linear(hidden, ffn, bias=False)
        self.down = nn.Linear(ffn, hidden, bias=False)
        self.norm = nn.LayerNorm(hidden)

    def forward(self, x):
        residual = x
        g = torch.nn.functional.silu(self.gate(x))
        u = self.up(x)
        h = self.down(g * u)
        return self.norm(h + residual)

model = FFNBlock(hidden=2048, ffn=8192).to(device=device, dtype=torch.float16)
x = torch.randn(16, 64, 2048, dtype=torch.float16, device=device)

def bench(fn, n=30):
    with torch.no_grad():
        for _ in range(3): fn(x)
        if device.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(n):
            fn(x)
            if device.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000

t_eager = bench(model)
print(f"Eager mode (no compilation):  {t_eager:.3f} ms")

try:
    model_compiled = torch.compile(model, mode='reduce-overhead')
    # Trigger compilation
    with torch.no_grad():
        _ = model_compiled(x)
    t_compiled = bench(model_compiled)
    print(f"torch.compile (reduce-overhead): {t_compiled:.3f} ms")
    print(f"Speedup: {t_eager/t_compiled:.2f}x")
    print()
    print("Note: torch.compile speedup varies. For LLM inference with custom")
    print("kernels (FlashAttention, fused GEMM), inference engines do better.")
except Exception as e:
    print(f"torch.compile: {e}")
    print(f"Eager mode only: {t_eager:.3f} ms")

Eager mode (no compilation):  2.203 ms
torch.compile (reduce-overhead): 1.818 ms
Speedup: 1.21x

Note: torch.compile speedup varies. For LLM inference with custom
kernels (FlashAttention, fused GEMM), inference engines do better.


## Try These Experiments

1. **Weight format sizes:** Load a real model config (e.g., from `transformers` — just the config, not weights). Calculate the expected FP16 weight size. How many safetensors shards would you need if each shard is 9.5 GB (a common convention)?

2. **Memory mapping vs full load:** Time `torch.load(path, map_location='cpu')` vs. loading with `mmap=True` for a model file. When does memory mapping help more — small or large models?

3. **ONNX operator coverage:** Export the `FFNBlock` to ONNX and count the number of MatMul, Add, and custom activation nodes. Then try adding a custom operation (like a rotary position embedding) that ONNX may not support natively — does the export fail?

## Key Takeaways

- **safetensors** is the production standard: safe (no code execution), fast (memory-mapped), shardable. Use it for all weight storage.
- **ONNX** bundles weights + computation graph for portability, but complex models with custom ops may not export cleanly. The industry is trending toward skipping ONNX and going PyTorch weights directly to inference engines.
- **torch.compile** provides automatic kernel fusion and selection — useful for standard architectures, limited for custom kernels.
- Cold start time is dominated by weight loading. Quantized weights (FP8, INT4) are faster to load because they are smaller.
- **What's next:** vLLM: PagedAttention & Continuous Batching — the most popular open-source inference engine.

## References
- *Inference Engineering* Ch 4.2 (pp. 101–105) — Deep Learning Frameworks and Libraries